# Fase 4: Sinais Alternativos e Sentimento de Notícias (FinBERT)

Este notebook demonstra o pipeline de NLP Financeiro com FinBERT e cálculo da métrica de **Divergência Preço-Sentimento**:
1. **FinBERT Batch Extractor**: Cálculo de probabilidades $(P_+, P_-, P_0)$ e score contínuo $S_{\text{news}} = P_+ - P_-$.
2. **Divergência Preço-Sentimento**: Padronização por Z-score móvel de 5 dias:
   $$\text{Divergence}_t = Z(\text{Sentiment}_t) - Z(\text{Price\_Return}_t)$$
3. **Sinalização Sistemática**: Alertas para divergências extremas ($|\text{Divergence}| \ge 2.0\sigma$).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.features.sentiment import (
    FinBertSentimentExtractor,
    compute_sentiment_divergence,
)

In [2]:
extractor = FinBertSentimentExtractor(use_fallback=True)

news_data = [
    "NVIDIA reports record quarterly revenue and beats AI chip estimates",
    "Central bank signals interest rates will remain steady throughout the quarter",
    "Tech giant faces antitrust lawsuit and warns of significant margin compression",
    "Biotech firm surges 20% on FDA approval of breakthrough oncology drug",
    "Retail sales miss expectations as consumer spending slumps",
]

df_sentiment = extractor.predict_sentiment_batch(news_data)
df_sentiment

In [3]:
# Simulação de Preço vs Sentimento Diário ao longo de 30 dias
np.random.seed(42)
dates = pd.date_range("2026-02-01", periods=30, freq="B")
base_price = 100.0 * np.exp(np.cumsum(np.random.normal(0, 0.01, size=30)))
sentiment_vals = np.random.normal(0, 0.2, size=30)

# Inserir divergência artificial no dia 15 (preço cai mas notícias são ultra positivas)
base_price[15] *= 0.93
sentiment_vals[15] = 0.85

price_s = pd.Series(base_price, index=dates)
sent_s = pd.Series(sentiment_vals, index=dates)

df_div = compute_sentiment_divergence(price_s, sent_s, window=5, threshold=2.0)
df_div[['price', 'sentiment', 'divergence', 'signal']].iloc[12:18]